# 02 · Train backbones & save per-scan scores  [GPU]
Trains each final backbone on the TRAIN fruits' pooled slices (all days), early stopping on validation AUC, light fine-tuning. Saves per-scan scores (grouped by fruit) for val & test so notebook 03 needs no GPU.

Set `RUN_HPO=True` to search hyper-parameters with Optuna (target = val AUC).

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()/'nbpkg'))
import numpy as np, pandas as pd
from config import CFG
import dataset as ds, eval_core as ec
CFG.out_dir.mkdir(parents=True, exist_ok=True)
print('data_root :', CFG.data_root); print('fruit_key :', CFG.fruit_key)

In [ ]:
import citrus_dl as dl, json, pickle
dl.set_seeds(CFG.seed)
fruits = dl.index_fruits(CFG); by={f.fruit_id:f for f in fruits}
sp=pd.read_csv(CFG.out_dir/'split_single.csv').set_index('fruit_id')['split']
tr_f=[by[i] for i in sp[sp=='train'].index]; va_f=[by[i] for i in sp[sp=='val'].index]
te_f=[by[i] for i in sp[sp=='test'].index]
print(f'train={len(tr_f)} val={len(va_f)} test={len(te_f)} fruits')

In [ ]:
RUN_HPO=False
DEFAULTS={'MobileNetV2':{'lr':5.6e-5,'dropout':0.25,'dense':256},
          'NASNetMobile':{'lr':1e-4,'dropout':0.30,'dense':128},
          'DenseNet121':{'lr':1e-4,'dropout':0.30,'dense':128},
          'InceptionV3':{'lr':2e-4,'dropout':0.40,'dense':64}}
hparams={}
for bb in CFG.final_backbones:
    hparams[bb]=dl.run_optuna(CFG,bb,tr_f,va_f)[0] if RUN_HPO else DEFAULTS[bb]
json.dump(hparams,open(CFG.out_dir/'hparams.json','w'),indent=2); hparams

In [ ]:
all_scores={}
for bb in CFG.final_backbones:
    print(f'\n=== {bb} ==='); m=dl.train_backbone(CFG,bb,tr_f,va_f,hparams[bb])
    m.save(CFG.out_dir/f'model_{bb}.keras')
    all_scores[bb]={'val':dl.predict_scan_scores(m,CFG,bb,va_f),
                    'test':dl.predict_scan_scores(m,CFG,bb,te_f)}
    import tensorflow as tf; tf.keras.backend.clear_session()
pickle.dump(all_scores,open(CFG.out_dir/'scan_scores.pkl','wb'))
print('saved scan_scores.pkl')